---
title: "VISUALIZATION"
format:
  html:
    embed-resources: true
    toc: true
jupyter: python3
---


In [1]:
#| output: false
import plotly.io as pio 
pio.renderers.default = "iframe"
import importlib
import pre_processing as prep
import sqlite3
import json
import datetime
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

conn = prep.connect_db("numero_data.sqlite")

# Load raw json
df_raw = prep.load_raw_sales(conn)
df_sales = prep.flatten_sales_json(df_raw)
prep.save_df_to_csv(df_sales,"sales_processed.csv")

# Dataset
df_meta, df_titles = prep.load_metadata_and_titles(conn) 
df = prep.join_sales_metadata(df_sales, df_meta, df_titles)
df_run, longest, segments, tmp, film_dates = prep.extract_longest_continuous_run(df, gap_days=7)

Loaded 100 raw film records.


Flattening sales JSON: 100%|██████████| 100/100 [00:00<00:00, 172.99it/s]


Successfully processed 65947 rows.


## Seasonality and Event Dynamics

To get a first sense of the dataset, I started by looking at total demand across the year. This line chart compares total, weekend, and weekday gross by month. 

In [2]:
df["weekend_gross_component"] = df["gross_today"].where(df["is_weekend_numero"], 0)

monthly = (
    df.groupby("month", as_index=False)
      .agg(total_gross=("gross_today", "sum"),
           weekend_gross=("weekend_gross_component", "sum")))

monthly["weekday_gross"] = monthly["total_gross"] - monthly["weekend_gross"]
monthly = monthly.sort_values("month")

total_M   = monthly["total_gross"]   / 1e6
weekend_M = monthly["weekend_gross"] / 1e6
weekday_M = monthly["weekday_gross"] / 1e6

month_labels = monthly["month"].dt.strftime("%b")

monthly_long = monthly.assign(
    Total=total_M,
    Weekend=weekend_M,
    Weekday=weekday_M,
)[["month", "Total", "Weekend", "Weekday"]]

monthly_long = monthly_long.melt(
    id_vars="month",
    value_vars=["Total", "Weekend", "Weekday"],
    var_name="Category",
    value_name="Gross_M"
)

monthly_long["month_label"] = monthly_long["month"].dt.strftime("%b")

fig = px.line(
    monthly_long,
    x="month_label",
    y="Gross_M",
    color="Category",
    markers=True,
    title="Monthly Indian Box Office Gross in Australia (2025)",
    labels={"Gross_M": "Gross ($ millions)", "month_label": "Month"}
)

fig.update_traces(
    hovertemplate="%{y:.2f}M" )

fig.update_layout(
    hovermode="x unified",
    hoverlabel=dict(bgcolor="white", font_size=13))

-> There is a clear spike in the middle of the year, especially in August, and most of that lift comes from the weekend line. This shows that Indian demand is strongly seasonal and very dependent on weekends.\
(As a note, “weekend” here is defined as Friday, Saturday and Sunday.)

Next, I broke the monthly trend into how many films we release and how strong they are. In this combo chart, the bars show the number of films per month, and the red line shows the median daily gross per film. \
-> Around July and August, we do not just release more titles, those films also earn more per day. This supports the idea that mid year is a highly competitive release window.

In [3]:
# Films count with median daily gross by month
films_per_month = (
    df.groupby("month")["title"]
      .nunique()
      .reset_index(name="num_films")
      .sort_values("month")
)

daily_month = (
    df.groupby(["month", "actual_sales_date"], as_index=False)["gross_today"]
      .sum()
      .rename(columns={"gross_today": "daily_gross"})
)

median_gross_month = (
    daily_month.groupby("month", as_index=False)["daily_gross"]
      .median()
      .rename(columns={"daily_gross": "median_daily_gross"})
)

month_stats = films_per_month.merge(median_gross_month, on="month", how="left")
month_stats = month_stats.sort_values("month")

month_labels = month_stats["month"].dt.strftime("%b")
x_pos = np.arange(len(month_stats))

num_films = month_stats["num_films"].values
median_gross_M = (month_stats["median_daily_gross"] / 1e6).values

# Add month label
month_stats = month_stats.sort_values("month")
month_stats["month_label"] = month_stats["month"].dt.strftime("%b")

month_stats["median_M"] = month_stats["median_daily_gross"] / 1e6

In [4]:
# Plot
bar_px = px.bar(
    month_stats,
    x="month",
    y="num_films",
    labels={"month": "Month", "num_films": "Number of films"},
    color_discrete_sequence=["cornflowerblue"],
)

line_px = px.line(
    month_stats,
    x="month",
    y="median_M",
    markers=True,
    labels={"median_M": "Median daily gross ($ millions)"},
    color_discrete_sequence=["crimson"],
)

# Subplot
fig = make_subplots(specs=[[{"secondary_y": True}]])

for trace in bar_px.data:
    # show month label once in unified hover
    trace.update(
        hovertemplate="Number of films: %{y}<extra></extra>"
    )
    fig.add_trace(trace, secondary_y=False)

# Add line traces from px (median gross)
for trace in line_px.data:
    trace.update(
        hovertemplate="Median daily gross: %{y:.2f}M<extra></extra>",
        line=dict(width=2),
        marker=dict(size=8)
    )
    fig.add_trace(trace, secondary_y=True)


fig.update_layout(
    title="Films Count and Median Daily Gross by Month (2025)",
    template="plotly_white",
    width=1000,
    height=520,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified"
)

fig.update_xaxes(
    tickformat="%b",
    tickmode="array",
    tickvals=month_stats["month"],
    ticktext=month_stats["month_label"]
)

fig.update_yaxes(title_text="Number of films", secondary_y=False)
fig.update_yaxes(title_text="Median daily gross ($ millions)", secondary_y=True)

fig.show()

To understand behaviour within the week, this bar chart shows total gross by day of week. \
-> Revenue begins to rise on Thursday, reaches its highest point on Friday and Saturday, and then softens on Sunday. This pattern confirms that Friday and Saturday are the core commercial days where screens and audience attention matter most. This also fits with common industry practice, since new films typically open on Friday, making it the first full day of audience demand and the natural peak of weekly cinema activity.

In [5]:
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

dow_gross = (
    df.groupby("dow")["gross_today"]
      .sum()
      .reindex(range(7))      
      .reset_index(name="total_gross")
)
dow_gross["total_gross_M"] = dow_gross["total_gross"] / 1e6

# Bar chart
fig = px.bar(
    dow_gross,
    x="dow",
    y="total_gross_M",
    labels={"dow": "Day of Week", "total_gross_M": "Total Gross ($ millions)"},
    color_discrete_sequence=["cornflowerblue"],
)

# format x ticks
fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(7)),
    ticktext=dow_labels
)

# tidy hover and layout
fig.update_traces(
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Total gross: %{y:.2f}M<br>"
        "<extra></extra>"
    )
)

fig.update_layout(
    title="Total Gross by Day of Week",
    template="plotly_white",
    width=800,
    height=480,
    hoverlabel=dict(bgcolor="white", font_size=13)
)

fig.show()


To understand behaviour within the week, this bar chart shows total gross by day of week. Revenue rises from Thursday, peaks on Friday and Saturday, and softens on Sunday. This confirms that Friday and Saturday are the key days where screens and audience attention matter most.

In [6]:
# Weekly trend
weekly_gross = df.groupby('week_start_date')['gross_today'].sum().reset_index()

# convert to millions
weekly_gross["gross_M"] = weekly_gross["gross_today"] / 1e6

iso = weekly_gross["week_start_date"].dt.isocalendar() 
weekly_gross["week_no"] = iso["week"].astype(str)

# Plot
fig = px.line(
    weekly_gross,
    x="week_start_date",
    y="gross_M",
    markers=True,
    title="Weekly Revenue",
    labels={"week_start_date": "Week Start", "gross_M": "Total Gross ($ millions)"},
    color_discrete_sequence=["#b22222"]
)

# Hover
fig.update_traces( 
    hovertemplate=("Week: %{customdata[0]}<br>" "Gross: %{y:.2f}M<extra></extra>" ), 
    customdata=weekly_gross[["week_no"]], 
    line=dict(width=2.5), 
    marker=dict(size=6) )

# x axis
fig.update_xaxes(
    dtick="M1",
    tickformat="%b",
    ticklabelmode="period"
)

# y axis
fig.update_yaxes(tickformat=".0f")

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=480,
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()


## Market Without Blockbusters

Since the industry is often shaped by a few major blockbusters, I wanted to avoid making any biased assumptions based on those titles alone. To do that, I began by examining the full distribution of film performance rather than jumping straight to the biggest hits. The boxplot of total gross per film makes the imbalance in the market very clear.\
-> Most titles sit tightly in a low range, while a long tail of outliers stretches far above the median. This shows that the typical film earns modestly, and only a small group of titles is responsible for lifting the entire market.

In [7]:
# Boxplot
film_gross = (
    df.groupby("title", as_index=False)["gross_today"].sum()
      .rename(columns={"gross_today": "total_gross"})
)
film_gross["total_gross"] = film_gross["total_gross"] / 1e6  # convert to millions

# Plot
fig = px.box(
    film_gross,
    x="total_gross",         
    points="all",           
    hover_data=["title"],   
    labels={"total_gross": "Total Gross per Film ($ million)"},
    title="Boxplot of Indian Film Performance",
    color_discrete_sequence=["cornflowerblue"]
)

fig.update_traces(
    hovertemplate="<b>%{x:.2f}M</b><br>Film: %{customdata[0]}<extra></extra>",
    marker=dict(opacity=0.8, size=6),
    boxmean=True
)

fig.update_layout(
    template="plotly_white",
    width=700,
    height=420,
    yaxis=dict(visible=False),   # hide empty y-axis (no categorical axis)
    margin=dict(l=40, r=20, t=60, b=40),
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

To understand who these heavy hitters are, the next chart isolates the top 12 films by total gross. Titles such as Kantara A Legend: Chapter 1 and Saiyaara rise far above the rest, confirming that a handful of releases are doing most of the work in the 2025 Indian box office. This sets up the next step of the analysis, where I remove these outliers to see how the market behaves without the influence of blockbuster noise.

In [8]:
# Top performing films (by revenue)
film_perf = (
    df.groupby("title", as_index=False)["gross_today"]
      .sum()
      .rename(columns={"gross_today": "total_gross"})
      .sort_values("total_gross", ascending=False)
      .head(12)
)

# Convert to millions and compute percent of total
film_perf["total_gross_M"] = film_perf["total_gross"] / 1e6
grand_total = film_gross["total_gross"].sum()  # film_gross from your earlier step (in millions)
film_perf["percent_of_total"] = (film_perf["total_gross_M"] / grand_total) * 100

# Text label
film_perf["pct_text"] = film_perf["percent_of_total"].map(lambda x: f"{x:.1f}%")

# Plot
fig = px.bar(
    film_perf,
    x="total_gross_M",
    y="title",
    orientation="h",
    text="pct_text",
    labels={"total_gross_M": "Total Gross ($ millions)", "title": "Film"},
    color_discrete_sequence=["cornflowerblue"],
    title="Top 12 Indian Films by Total Gross in Australia"
)

# Show the percent text inside bars
fig.update_traces(
    textposition="inside",
    textfont=dict(color="white", size=12),
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Total gross: %{x:.2f}M<br>"
        "Share of total: %{customdata[0]:.1f}%<extra></extra>"
    ),
    customdata=film_perf[["percent_of_total"]].values
)

# Put the largest film at the top
fig.update_yaxes(autorange="reversed")

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=600,
    margin=dict(l=220, r=40, t=70, b=40),
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

In [9]:
top_12_titles = film_perf['title'].tolist()
top_films_data = df[df['title'].isin(top_12_titles)]
film_dates = (
    top_films_data.groupby('title')['actual_sales_date']
    .agg(['min', 'max'])
    .reset_index()
    .rename(columns={'min': 'release_date', 'max': 'last_showing'})
)
film_perf_with_dates = film_perf.merge(film_dates, on='title', how='left')
display(film_perf_with_dates[['title', 'total_gross', 'release_date', 'last_showing', 'percent_of_total']])

,title,total_gross,release_date,last_showing,percent_of_total
0,Kantara A Legend: Chapter 1,182044184.0,2025-10-02,2025-11-12,5.821928
1,Saiyaara,176392024.0,2025-07-17,2025-09-10,5.641167
2,Chhaava,167509548.0,2025-02-13,2025-04-02,5.357098
3,Coolie,144641071.0,2025-08-14,2025-09-03,4.625744
4,L2: Empuraan,140466294.0,2025-03-27,2025-04-16,4.492231
5,Sardaar Ji 3,121291180.0,2025-06-26,2025-07-30,3.878995
6,Chal Mera Putt 4,117511756.0,2025-07-31,2025-09-10,3.758126
7,Sitaare Zameen Par,110568851.0,2025-06-19,2025-08-20,3.536086
8,War 2,105243152.0,2025-08-14,2025-09-10,3.365765
9,Lokah Chapter One: Chandra,95631415.0,2025-08-28,2025-10-08,3.058374


To avoid biased conclusions driven by a few blockbusters, I removed the top 12 films by total gross and reanalyzed the same weekly and monthly metrics. This is a simple counterfactual test that compares the market with and without the biggest outliers to show how much they distort the overall picture.

In [10]:
# Excluding top 12
top_12_ids = df.groupby('numero_film_id')['gross_today'].sum().sort_values(ascending=False).head(12).index.tolist()
sales_mid_market = df[~df['numero_film_id'].isin(top_12_ids)]

In [11]:
total_market_gross = df['gross_today'].sum()
mid_market_gross = sales_mid_market['gross_today'].sum()
drop_off = (1 - (mid_market_gross / total_market_gross)) * 100

print(f"Without Top 12 films, we removed {drop_off:.1f}% of the total revenue.")
print(f"Original Market Size: ${total_market_gross:,.0f}")
print(f"Remaining Market Size: ${mid_market_gross:,.0f}")

Without Top 12 films, we removed 48.9% of the total revenue.
Original Market Size: $3,126,871,244
Remaining Market Size: $1,599,215,276


The results show that removing the top 12 films eliminated nearly 50% of total revenue. The weekly revenue series that once showed dramatic spikes now becomes largely flat. The large peaks in March, August, and October disappear or shrink dramatically and the baseline between peaks falls to a much lower level.

In [12]:
# Daily aggregates
daily_total = df.groupby('actual_sales_date', as_index=False)['gross_today'].sum()
daily_excl_outliers = (
    df[~df['numero_film_id'].isin(top_12_ids)]
      .groupby('actual_sales_date', as_index=False)['gross_today'].sum()
)

daily_total['actual_sales_date'] = pd.to_datetime(daily_total['actual_sales_date'])
daily_excl_outliers['actual_sales_date'] = pd.to_datetime(daily_excl_outliers['actual_sales_date'])
daily_total = daily_total.sort_values('actual_sales_date')
daily_excl_outliers = daily_excl_outliers.sort_values('actual_sales_date')
daily_total['gross_M'] = daily_total['gross_today'] / 1e6
daily_excl_outliers['gross_M'] = daily_excl_outliers['gross_today'] / 1e6

fig = go.Figure()

# All films
fig.add_trace(go.Scatter(
    x=daily_total['actual_sales_date'],
    y=daily_total['gross_M'],
    mode='lines',
    name='All Films',
    line=dict(color='lightgray'),
    opacity=0.8,
    hovertemplate="<b>%{x|%b %d, %Y}</b><br>All films: %{y:.2f}M<extra></extra>"
))

# Excluding top 12
fig.add_trace(go.Scatter(
    x=daily_excl_outliers['actual_sales_date'],
    y=daily_excl_outliers['gross_M'],
    mode='lines+markers',
    name='Excluding Top 12',
    line=dict(color='#2e8b57', width=2),
    marker=dict(size=4),
    hovertemplate="<b>%{x|%b %d, %Y}</b><br>Excl top 12: %{y:.2f}M<extra></extra>"
))

# x axis
fig.update_xaxes(
    dtick="M1",
    tickformat="%b",
    ticklabelmode="period"
)

# y axis
fig.update_yaxes(title_text="Total Gross ($ millions)", tickformat=".0f")

fig.update_layout(
    title="Performance Without the Top 12",
    template="plotly_white",
    width=1100,
    height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

-> This change tells a clear story. The apparent volatility and the large seasonal spikes are not the result of steady shifts in audience habit. They are the result of a small number of event films. In other words, the market is event driven rather than habit driven. When a blockbuster opens, it creates a large temporary surge. When no blockbuster is present, the market reverts to a low baseline.

This finding has important implications for both planning and operations. If we rely on simple trend models that ignore outliers, we will overestimate the true baseline and underestimate how volatile the market really is. Blockbusters behave like separate events, and forecasts need to treat them that way.

From an operational point of view, the industry can run more effectively in two different modes. During event windows, competition for screens is intense and the market surges. During non event windows, the average film struggles to attract audiences and the baseline drops sharply. Understanding which mode we are in is essential for accurate forecasting, smarter scheduling, and better allocation of screens.

Even though hits dominate the peaks, I suspected there might still be underlying seasonality.

-> ACF and PACF tests confirm repeated patterns, showing that while blockbusters drive the peaks, the baseline market still follows a seasonal rhythm worth exploring further.

In [13]:
import statsmodels.api as sm

daily_excl_outliers["actual_sales_date"] = pd.to_datetime(daily_excl_outliers["actual_sales_date"])
ts = daily_excl_outliers.set_index("actual_sales_date")["gross_today"].sort_index()

# Compute ACF and PACF with 95% CI
nlags = 30
acf_vals, acf_confint = sm.tsa.stattools.acf(ts, nlags=nlags, alpha=0.05)
pacf_vals, pacf_confint = sm.tsa.stattools.pacf(ts, nlags=nlags, alpha=0.05, method="ywm")

lags = np.arange(len(acf_vals))

# Convert confint arrays to lower/upper
acf_lower = acf_confint[:, 0] - acf_vals
acf_upper = acf_confint[:, 1] - acf_vals
pacf_lower = pacf_confint[:, 0] - pacf_vals
pacf_upper = pacf_confint[:, 1] - pacf_vals

# Figure with 2 rows
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("Autocorrelation (ACF) - Daily Gross", "Partial Autocorrelation (PACF) - Daily Gross"))

# ACF
fig.add_trace(go.Bar(x=lags, y=acf_vals, marker_color="#1f77b4", name="ACF", hovertemplate="Lag %{x}<br>ACF: %{y:.3f}<extra></extra>"), row=1, col=1)
# CI band
fig.add_trace(go.Scatter(
    x=np.concatenate([lags, lags[::-1]]),
    y=np.concatenate([acf_confint[:,1], acf_confint[::-1,0]]),
    fill="toself",
    fillcolor="rgba(31,119,180,0.15)",
    line=dict(color="rgba(255,255,255,0)"),
    hoverinfo="skip",
    showlegend=False
), row=1, col=1)
# zero line
fig.add_shape(type="line", x0=0, x1=nlags, y0=0, y1=0, line=dict(color="black", dash="dash"), row=1, col=1)

# PACF
fig.add_trace(go.Bar(x=lags, y=pacf_vals, marker_color="#d62728", name="PACF", hovertemplate="Lag %{x}<br>PACF: %{y:.3f}<extra></extra>"), row=2, col=1)
fig.add_trace(go.Scatter(
    x=np.concatenate([lags, lags[::-1]]),
    y=np.concatenate([pacf_confint[:,1], pacf_confint[::-1,0]]),
    fill="toself",
    fillcolor="rgba(214,39,40,0.12)",
    line=dict(color="rgba(255,255,255,0)"),
    hoverinfo="skip",
    showlegend=False
), row=2, col=1)
fig.add_shape(type="line", x0=0, x1=nlags, y0=0, y1=0, line=dict(color="black", dash="dash"), row=2, col=1)

fig.update_xaxes(title_text="Lag (days)", row=2, col=1, dtick=1)
fig.update_yaxes(title_text="Correlation Coefficient", row=1, col=1)
fig.update_yaxes(title_text="Correlation Coefficient", row=2, col=1)

fig.update_layout(
    height=800,
    width=1000,
    template="plotly_white",
    title_text="ACF and PACF for Daily Gross (excluding top 12)",
    hovermode="x"
)

fig.show()

Next, I looked at who benefits most from this structure by looking at Top 10 Distributors. The data shows Mindblowing Films is the dominant distributor.

## Distributor Performance and Circuit Concentration

In [14]:
# Top 10 Distributors
dist_market_share = (
    df.groupby("distributor", as_index=False)["gross_today"]
      .sum()
      .sort_values("gross_today", ascending=False)
)

dist_market_share["gross_M"] = dist_market_share["gross_today"] / 1e6
top10 = dist_market_share.head(10).copy()

# label for inside bar text
top10["label"] = top10["gross_M"].map(lambda v: f"{v:.1f}M")

# Plot
fig = px.bar(
    top10,
    x="gross_M",
    y="distributor",
    orientation="h",
    text="label",
    labels={"gross_M": "Total Gross ($ millions)", "distributor": ""},
    color_discrete_sequence=["cornflowerblue"],
    title="Top 10 Distributors"
)

fig.update_yaxes(autorange="reversed")

# hover
fig.update_traces(
    textposition="inside",
    textfont=dict(color="white", size=12),
    hovertemplate="<b>%{y}</b><br>Total gross: %{x:.2f}M<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=520,
    margin=dict(l=220, r=40, t=70, b=40),
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

In [15]:
# Total gross and film count per distributor
dist_stats = (
    df.groupby(["distributor", "title"], as_index=False)["gross_today"]
      .sum()
      .rename(columns={"gross_today": "film_gross"})
      .groupby("distributor", as_index=False)
      .agg(
          total_gross=("film_gross", "sum"),
          num_films=("title", "nunique"),
          avg_gross_per_film=("film_gross", "mean")
      )
      .sort_values("total_gross", ascending=False)
)

dist_stats["avg_gross_per_film"] = dist_stats["avg_gross_per_film"] / 1e6
dist_stats_top = dist_stats.head(10)
dist_stats["total_gross"] = dist_stats["total_gross"] / 1e6
grand_total = dist_stats["total_gross"].sum()
dist_stats_top["percent_of_total"] = (dist_stats["total_gross"] / grand_total) *100

display(dist_stats_top[["distributor", "total_gross", "percent_of_total", "avg_gross_per_film"]]
          .round({"total_gross": 2, "percent_of_total": 2, "avg_gross_per_film":2}))

,distributor,total_gross,percent_of_total,avg_gross_per_film
3,Mindblowing,846236394.0,27.51,49.78
1,Forum Distribution,458183666.0,14.90,30.55
7,Tolly Movies,411821712.0,13.39,27.45
0,Cyber Systems,394466536.0,12.83,35.86
2,Home Screen Entertainment,215413384.0,7.00,17.95
12,Zstars Entertainment,210804260.0,6.85,52.70
10,White Hill,165123939.0,5.37,55.04
4,Moviegoers Entertainment,145331031.0,4.73,24.22
9,Wanderlust Films,120813485.0,3.93,30.20
11,Zee Studios,65078800.0,2.12,9.30


And even after removing the top 12 films, they still remain number one.
However, without the outliers, Tolly Movies becomes much closer to Mindblowing.

In [16]:
sales_mid_market = df[~df["numero_film_id"].isin(top_12_ids)]
dist_mid_share = (
    sales_mid_market.groupby("distributor", as_index=False)["gross_today"]
    .sum()
    .sort_values("gross_today", ascending=False)
    .head(10)
)

# Choose units and create plotting columns
max_val = dist_mid_share["gross_today"].max()
dist_mid_share["value_plot"] = dist_mid_share["gross_today"] / 1e6
x_label = "Total Gross ($ millions)"
dist_mid_share["label_text"] = dist_mid_share["value_plot"].map(lambda v: f"{v:.1f}M")

# Plot
fig = px.bar(
    dist_mid_share,
    x="value_plot",
    y="distributor",
    orientation="h",
    text="label_text",
    labels={"value_plot": x_label, "distributor": ""},
    color_discrete_sequence=["cornflowerblue"],
    title="Top 10 Distributors (Excluding Top 12 Hits)"
)

# Hover
fig.update_traces(
    textposition="inside",
    textfont=dict(color="white", size=12),
    hovertemplate="<b>%{y}</b><br>Total gross: %{x:.2f}<extra></extra>"
)

fig.update_yaxes(autorange="reversed")

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=520,
    margin=dict(l=220, r=40, t=70, b=40),
    hoverlabel=dict(bgcolor="white", font_size=12)
)

fig.show()

If state and city patterns tell us where audiences live, the distributor by circuit view shows how those audiences are reached. The heatmap below shows which distributors earn most with which cinema circuits. 

-> Mindblowing, Tolly Movies and Forum Distribution dominate Hoyts and also perform strongly at Event and Village, while circuits like United, Reading and ICA contribute very little for most distributors. 

-> **Recommendation**: Prioritise targeted partnerships and pilots with the circuits that already convert well for each distributor, and run controlled tests in underperforming circuits to determine whether low revenue is due to supply or demand. Treat concentrated wins as both an opportunity and a risk because they can deliver fast returns but create exposure if a single circuit underperforms, so pair aggressive circuit plays with diversification tests and a net contribution check.

In [17]:
# Distributor by Circuit
dist_circuit = (
    sales_mid_market
    .groupby(['distributor', 'circuit_name'], as_index=False)['gross_today']
    .sum())
dist_circuit['gross_M'] = dist_circuit['gross_today'] / 1e6

# Top distributors and circuits
top_dists = dist_circuit.groupby('distributor')['gross_M'].sum().nlargest(10).index
top_circuits = dist_circuit.groupby('circuit_name')['gross_M'].sum().nlargest(6).index

filtered_map = dist_circuit[
    dist_circuit['distributor'].isin(top_dists) &
    dist_circuit['circuit_name'].isin(top_circuits)].copy()

pivot_map = filtered_map.pivot(index='distributor', columns='circuit_name', values='gross_M').fillna(0)
dist_totals = pivot_map.sum(axis=1).sort_values(ascending=False).index
circuit_totals = pivot_map.sum(axis=0).sort_values(ascending=False).index
pivot_map_sorted = pivot_map.loc[dist_totals, circuit_totals]

# Text annotations
text_vals = pivot_map_sorted.round(1).astype(str).values

# Plot
fig = go.Figure(data=go.Heatmap(
    z=pivot_map_sorted.values,
    x=pivot_map_sorted.columns.tolist(),
    y=pivot_map_sorted.index.tolist(),
    text=text_vals,
    texttemplate="%{text}",
    colorscale="Greens",
    colorbar=dict(title="Gross ($M)"),
    hovertemplate="<b>%{y}</b><br>%{x}: %{z:.1f}M<extra></extra>"
))
fig.update_yaxes(autorange="reversed")
fig.update_layout(
    title="Distributor Gross by Circuit",
    xaxis_title="Cinema Circuit",
    yaxis_title="Distributor",
    template="plotly_white",
    width=1000,
    height=600,
    margin=dict(l=220, r=40, t=70, b=80)
)
fig.show()

## Theatre level: volume and efficiency

Building on this idea of regional variation, I then moved from distributors to the locations themselves and examined how individual theatres perform in terms of volume and efficiency.

The bubble chart shows three dimensions of performance across states, cities and theatres. The horizontal axis shows volume, measured as the sum of each film’s longest continuous run days at the location. The vertical axis shows efficiency, measured as average daily revenue computed as total revenue divided by run‑based days. Bubble size reflects total revenue and colour indicates state.

2 distinct opportunity types emerge:

- **Premium event screens (IMAX) sit in the upper left:** they post extremely high average daily revenue but run far fewer days. However, they also come with higher costs and limited flexibility. Therefore, these screens are best used for tentpoles and premium releases rather than regular expansion.

- **Scalable high-efficiency theatres sit in the upper right:** they combine high daily revenue with long runs and therefore offer the best, repeatable upside from adding days or improving showtimes.

-> **Recommendation**: Prioritise adding screens and improving showtimes at high efficiency theatres, treat premium formats as a separate planning bucket for tentpoles, and focus further analysis on upper right theatres to identify repeatable patterns that make them successful.

In [18]:
# Metrics
cinema_stats = df_run.groupby(['state','city','theatre_name' ]).agg(
    total_gross=('gross_today', 'sum'),
    days_active=('actual_sales_date', 'nunique')
).reset_index()

cinema_stats['avg_daily_gross'] = cinema_stats['total_gross'] / cinema_stats['days_active']
# Filter out
cinema_stats = cinema_stats[cinema_stats['total_gross'] > 1000]
cinema_stats = cinema_stats[cinema_stats['state'].isin(['New South Wales (inc ACT)','Victoria (inc TAS)'])]
# Plot
fig = px.scatter(cinema_stats, x="days_active", y="avg_daily_gross", size="total_gross", color="state",
    # Hover
    hover_name="theatre_name",
    hover_data={
        "city": True,
        "state": False,
        "days_active": True,
        "avg_daily_gross": ':$,.0f', 
        "total_gross": ':$,.0f'},

    title='<b>Volume and Efficiency</b>',
    labels={
        "days_active": "Life time (Days)",
        "avg_daily_gross": "Avg Daily Revenue",
        "state": "State",
        "total_gross": "Total Revenue"},
    template="plotly_white",
    size_max=60, 
    height=700  
)

# Median lines
median_efficiency = cinema_stats['avg_daily_gross'].median()
median_volume = cinema_stats['days_active'].median()

fig.add_hline(y=median_efficiency, line_dash="dash", line_color="gray", annotation_text="Median Efficiency")
fig.add_vline(x=median_volume, line_dash="dash", line_color="gray", annotation_text="Median Volume")

fig.show()

In [19]:
# Location
pd.set_option('display.max_rows', None)
cinema_stats = df.groupby(['state','city','theatre_name' ]).agg(
    total_gross=('gross_today', 'sum'),
    days_active=('actual_sales_date', 'nunique'),
    avg_daily_gross=('gross_today', 'mean')
).reset_index()
# cinema_stats = cinema_stats[cinema_stats['total_gross'] > 1000]
cinema_stats_sort = cinema_stats.sort_values('total_gross', ascending=False)
grand_total = cinema_stats_sort['total_gross'].sum()
cinema_stats_sort['percent_share'] = (cinema_stats_sort['total_gross'] / grand_total) * 100

display(
    cinema_stats_sort.head(10).style.format({
        'total_gross': '${:,.0f}',
        'avg_daily_gross': '${:,.0f}',
        'percent_share': '{:.2f}%'
    })
)

,state,city,theatre_name,total_gross,days_active,avg_daily_gross,percent_share
68,New South Wales (inc ACT),West and Blue Mountains,Blacktown,"$272,377,817",315,"$216,689",8.71%
202,Victoria (inc TAS),West Melbourne,Sunshine,"$209,822,774",315,"$104,545",6.71%
54,New South Wales (inc ACT),Parramatta & Ryde,Parramatta,"$186,267,643",315,"$175,890",5.96%
204,Victoria (inc TAS),West Melbourne,Werribee,"$130,211,836",315,"$104,840",4.16%
216,Western Australia,Perth - South East,Carousel,"$127,762,193",315,"$190,690",4.09%
190,Victoria (inc TAS),South East Melbourne,Chadstone,"$120,959,042",308,"$192,917",3.87%
92,Queensland,Brisbane - South,Garden City Mt. Gravatt,"$117,022,392",315,"$94,832",3.74%
192,Victoria (inc TAS),South East Melbourne,Fountain Gate,"$116,249,483",315,"$128,311",3.72%
137,South Australia,Adelaide - West,Hoyts Arndale,"$100,301,966",315,"$114,239",3.21%
147,Victoria (inc TAS),Central Inner Melbourne,Docklands,"$87,963,587",308,"$124,418",2.81%
